# Aralin 11 - Protokol ng Ahente-sa-Ahente (A2A)


## Pagsasaayos


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Ano ang A2A Protocol?

The **Agent-to-Agent (A2A) protocol** ay isang bukas na pamantayan na nagpapahintulot sa mga AI agent na makipag-ugnayan,
matuklasan ang isa't isa, at makipagtulungan — kahit na sila ay binuo sa iba't ibang frameworks o naka-host
ng iba't ibang services.

Key concepts:

- **Pagdiskubre** – Naglalathala ang mga ahente ng isang *Agent Card* na naglalarawan ng kanilang mga kakayahan, na ginagawang
  madali para sa ibang mga ahente (o mga orchestrator) na mahanap ang tamang espesyalista para sa isang gawain.
- **Pagpapasa ng Mensahe** – Nagpapalitan ang mga ahente ng mga istrukturadong mensahe sa pamamagitan ng isang karaniwang protocol, kaya ang
  kahilingan mula sa isang ahente ay maaaring maunawaan at matupad ng isa pa anuman ang panloob nitong
  implementasyon.
- **Buhay ng Gawain** – Tinutukoy ng A2A ang mga estado tulad ng *submitted*, *working*, *completed*, at
  *failed*, na nagbibigay sa orchestrator ng buong kakayahang makita kung paano umuusad ang isang ipinagawang gawain.

Sa araling ito sinisimulate namin ang kolaborasyon ayon sa istilong A2A sa pamamagitan ng pag-uugnay ng tatlong espesyalistang ahente sa paglalakbay
sa isang workflow kung saan ang bawat ahente ay nag-aambag ng kanyang kadalubhasaan at ipinapasa ang mga resulta sa susunod.


## Paglikha ng Mga Espesyalisadong Ahente ng Paglalakbay


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Pakikipagtulungan ng Maramihang Ahente sa Daloy ng Trabaho

Kinokonekta namin ang tatlong ahente sa isang sunud-sunod na daloy ng trabaho na sumasalamin sa A2A na pagpapasa ng mensahe:

1. **CurrencyExchangeAgent** tumatanggap ng kahilingan mula sa gumagamit at nagbibigay ng gabay tungkol sa palitan ng pera.
2. **ActivityPlannerAgent** tumatanggap ng pinayamang konteksto at nagdaragdag ng mga rekomendasyon para sa mga aktibidad.
3. **TravelManagerAgent** pinagsasama ang parehong mga input upang makabuo ng pangwakas na buod ng paglalakbay.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Pag-unawa sa A2A sa Produksyon

Sa isang kapaligiran ng produksyon, binubuksan ng protocol na A2A ang makapangyarihang mga senaryo ng pagtutulungan sa pagitan ng mga serbisyo:

| Kakayahan | Paglalarawan |
|---|---|
| **Pagtutulungan ng iba't ibang framework** | Ang isang ahente na binuo gamit ang isang framework ay maaaring magpasa ng mga gawain sa isang ahente na binuo gamit ang anumang ibang framework na sumusunod sa A2A, na nagpapahintulot ng tunay na pakikipag-ugnayan sa pagitan ng mga organisasyon. |
| **Hangganan ng serbisyo** | Maaaring tumakbo ang mga ahente sa magkakahiwalay na microservices, mga rehiyon ng cloud, o kahit sa iba't ibang mga organisasyon habang patuloy na nakikipagtulungan nang walang putol. |
| **Dinamiko na pagtuklas** | Maaaring mag-query ang isang orchestrator sa isang Agent Card registry habang tumatakbo (runtime) upang hanapin ang pinaka-angkop na espesyalista para sa isang partikular na sub-task. |
| **Streaming & push notifications** | Sinusuportahan ng A2A ang Server-Sent Events (SSE) para sa real-time na pag-update ng progreso at mga push notification para sa mga gawaing tumatagal nang matagal. |

Ang workflow na binuo namin sa itaas ay isang pinasimple, in-process na bersyon ng pattern na ito. Sa isang totoong deployment, maglalantad ang bawat ahente ng isang HTTP endpoint, maglalathala ng Agent Card, at makikipag-ugnayan sa pamamagitan ng A2A JSON-RPC protocol.


## Buod

Sa araling ito natutunan mo:

1. **Ano ang A2A na protokol** — isang bukas na pamantayan para sa pagtuklas sa pagitan ng mga ahente, pagmemensahe, at pamamahala ng mga gawain.
2. **Paano lumikha ng mga espesyalisadong ahente** — isang ahenteng Palitan ng Salapi, isang ahenteng Tagaplano ng Aktibidad, at isang Travel Manager na orchestrator.
3. **Paano iugnay ang mga ahente sa isang workflow** — gamit ang `WorkflowBuilder` upang i-modelo ang sunud-sunod na pagpapasa ng mga mensahe sa pagitan ng mga ahente.
4. **Paano gumagana ang A2A sa produksyon** — nagpapagana ng kolaborasyon sa pagitan ng iba't ibang framework at serbisyo gamit ang dinamikong pagtuklas at streaming na mga pag-update.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Paunawa**:
Ang dokumentong ito ay isinalin gamit ang serbisyong pagsasalin ng AI na [Co-op Translator](https://github.com/Azure/co-op-translator). Bagaman pinagsisikapan naming maging tumpak, pakitandaan na ang mga awtomatikong salin ay maaaring maglaman ng mga pagkakamali o hindi pagkakatumpak. Ang orihinal na dokumento sa likas nitong wika ang dapat ituring na opisyal na sanggunian. Para sa kritikal na impormasyon, inirerekomenda ang propesyonal na pagsasaling-tao. Hindi kami mananagot sa anumang hindi pagkakaunawaan o maling interpretasyon na maaaring magmula sa paggamit ng salin na ito.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
